In [0]:
from pyspark.sql.functions import rand, when, col
 
# Create base dataset (120 rows)
df = spark.range(1, 121).withColumnRenamed("id", "order_id")
 
# Add columns
df = df.withColumn("customer_name",
        when(rand() > 0.5, "Arjun")
        .otherwise("Meera")
)
 
df = df.withColumn("city",
        when(rand() < 0.25, "Kochi")
        .when(rand() < 0.5, "Chennai")
        .when(rand() < 0.75, "Bangalore")
        .otherwise("Hyderabad")
)
 
df = df.withColumn("product",
        when(rand() < 0.3, "Laptop")
        .when(rand() < 0.6, "Mobile")
        .otherwise("Tablet")
)
 
df = df.withColumn("quantity", (rand()*3 + 1).cast("int"))
 
df = df.withColumn("price", (rand()*50000 + 10000).cast("int"))
 
# Introduce NULL values
df = df.withColumn(
    "price",
    when(rand() < 0.1, None).otherwise(col("price"))
)
 
# Create revenue
df = df.withColumn("revenue", col("quantity") * col("price"))
 
df.show()

+--------+-------------+---------+-------+--------+-----+-------+
|order_id|customer_name|     city|product|quantity|price|revenue|
+--------+-------------+---------+-------+--------+-----+-------+
|       1|        Meera|Hyderabad| Mobile|       2|51593| 103186|
|       2|        Arjun|  Chennai| Mobile|       3|29445|  88335|
|       3|        Meera|Bangalore| Mobile|       3| NULL|   NULL|
|       4|        Arjun|  Chennai| Mobile|       1|28741|  28741|
|       5|        Arjun|Hyderabad| Tablet|       3|57449| 172347|
|       6|        Arjun|  Chennai| Mobile|       2|22859|  45718|
|       7|        Meera|Bangalore| Tablet|       2|29863|  59726|
|       8|        Arjun|  Chennai| Mobile|       3|11890|  35670|
|       9|        Meera|Hyderabad| Laptop|       3|29035|  87105|
|      10|        Arjun|Bangalore| Mobile|       2|58467| 116934|
|      11|        Meera|    Kochi| Mobile|       2|27572|  55144|
|      12|        Arjun|    Kochi| Mobile|       3|27039|  81117|
|      13|

In [0]:
df_dup = df.union(df.limit(20))   # add 20 duplicate rows
 
df_dup.show()

+--------+-------------+---------+-------+--------+-----+-------+
|order_id|customer_name|     city|product|quantity|price|revenue|
+--------+-------------+---------+-------+--------+-----+-------+
|       1|        Meera|Hyderabad| Mobile|       2|51593| 103186|
|       2|        Arjun|  Chennai| Mobile|       3|29445|  88335|
|       3|        Meera|Bangalore| Mobile|       3| NULL|   NULL|
|       4|        Arjun|  Chennai| Mobile|       1|28741|  28741|
|       5|        Arjun|Hyderabad| Tablet|       3|57449| 172347|
|       6|        Arjun|  Chennai| Mobile|       2|22859|  45718|
|       7|        Meera|Bangalore| Tablet|       2|29863|  59726|
|       8|        Arjun|  Chennai| Mobile|       3|11890|  35670|
|       9|        Meera|Hyderabad| Laptop|       3|29035|  87105|
|      10|        Arjun|Bangalore| Mobile|       2|58467| 116934|
|      11|        Meera|    Kochi| Mobile|       2|27572|  55144|
|      12|        Arjun|    Kochi| Mobile|       3|27039|  81117|
|      13|

# Task 1
- Remove duplicate records


In [0]:
df_dup.dropDuplicates().show()

+--------+-------------+---------+-------+--------+-----+-------+
|order_id|customer_name|     city|product|quantity|price|revenue|
+--------+-------------+---------+-------+--------+-----+-------+
|       7|        Meera|Bangalore| Tablet|       2|29863|  59726|
|       8|        Arjun|  Chennai| Mobile|       3|11890|  35670|
|       9|        Meera|Hyderabad| Laptop|       3|29035|  87105|
|      14|        Meera|    Kochi| Mobile|       3|32641|  97923|
|       2|        Arjun|  Chennai| Mobile|       3|29445|  88335|
|       3|        Meera|Bangalore| Mobile|       3| NULL|   NULL|
|      10|        Arjun|Bangalore| Mobile|       2|58467| 116934|
|      13|        Meera|  Chennai| Mobile|       3|34089| 102267|
|      11|        Meera|    Kochi| Mobile|       2|27572|  55144|
|       5|        Arjun|Hyderabad| Tablet|       3|57449| 172347|
|       4|        Arjun|  Chennai| Mobile|       1|28741|  28741|
|       1|        Meera|Hyderabad| Mobile|       2|51593| 103186|
|      12|

# Task 2
- Replace null values


In [0]:

df_dup.fillna({'price': 0, 'revenue': 0}).show()

+--------+-------------+---------+-------+--------+-----+-------+
|order_id|customer_name|     city|product|quantity|price|revenue|
+--------+-------------+---------+-------+--------+-----+-------+
|       1|        Meera|Hyderabad| Mobile|       2|51593| 103186|
|       2|        Arjun|  Chennai| Mobile|       3|29445|  88335|
|       3|        Meera|Bangalore| Mobile|       3|    0|      0|
|       4|        Arjun|  Chennai| Mobile|       1|28741|  28741|
|       5|        Arjun|Hyderabad| Tablet|       3|57449| 172347|
|       6|        Arjun|  Chennai| Mobile|       2|22859|  45718|
|       7|        Meera|Bangalore| Tablet|       2|29863|  59726|
|       8|        Arjun|  Chennai| Mobile|       3|11890|  35670|
|       9|        Meera|Hyderabad| Laptop|       3|29035|  87105|
|      10|        Arjun|Bangalore| Mobile|       2|58467| 116934|
|      11|        Meera|    Kochi| Mobile|       2|27572|  55144|
|      12|        Arjun|    Kochi| Mobile|       3|27039|  81117|
|      13|

# Task 3
- Create category column


In [0]:
df_dup.withColumn("category", when(col("product") =="Mobile", "electronics").otherwise("other")).show()

+--------+-------------+---------+-------+--------+-----+-------+-----------+
|order_id|customer_name|     city|product|quantity|price|revenue|   category|
+--------+-------------+---------+-------+--------+-----+-------+-----------+
|       1|        Meera|Hyderabad| Mobile|       2|51593| 103186|electronics|
|       2|        Arjun|  Chennai| Mobile|       3|29445|  88335|electronics|
|       3|        Meera|Bangalore| Mobile|       3| NULL|   NULL|electronics|
|       4|        Arjun|  Chennai| Mobile|       1|28741|  28741|electronics|
|       5|        Arjun|Hyderabad| Tablet|       3|57449| 172347|      other|
|       6|        Arjun|  Chennai| Mobile|       2|22859|  45718|electronics|
|       7|        Meera|Bangalore| Tablet|       2|29863|  59726|      other|
|       8|        Arjun|  Chennai| Mobile|       3|11890|  35670|electronics|
|       9|        Meera|Hyderabad| Laptop|       3|29035|  87105|      other|
|      10|        Arjun|Bangalore| Mobile|       2|58467| 116934

# Task 4
- Build full pipeline: clean + transform + show output

In [0]:
df_c = df_dup.na.drop()
df_c = df_c.dropDuplicates()
df_c = df_c.fillna({'price': 0, 'revenue': 0})
df_clean = df_c.withColumn("category", when(col("product") == "Mobile", "electronics").otherwise("other"))
df_clean.select("order_id", "customer_name", "city", "product","revenue").show()

+--------+-------------+---------+-------+-------+
|order_id|customer_name|     city|product|revenue|
+--------+-------------+---------+-------+-------+
|       7|        Meera|Bangalore| Tablet|  59726|
|       8|        Arjun|  Chennai| Mobile|  35670|
|       9|        Meera|Hyderabad| Laptop|  87105|
|      14|        Meera|    Kochi| Mobile|  97923|
|       2|        Arjun|  Chennai| Mobile|  88335|
|      10|        Arjun|Bangalore| Mobile| 116934|
|      13|        Meera|  Chennai| Mobile| 102267|
|      11|        Meera|    Kochi| Mobile|  55144|
|       5|        Arjun|Hyderabad| Tablet| 172347|
|       4|        Arjun|  Chennai| Mobile|  28741|
|       1|        Meera|Hyderabad| Mobile| 103186|
|      12|        Arjun|    Kochi| Mobile|  81117|
|       6|        Arjun|  Chennai| Mobile|  45718|
|      15|        Arjun|  Chennai| Mobile|  15466|
|      17|        Meera|  Chennai| Mobile|  87218|
|      19|        Arjun|    Kochi| Mobile|  18764|
|      22|        Arjun|Bangalo